# 01 — Tesseract OCR Pipeline

Runs Tesseract on the 17 Veenhof chapter PDFs. Uses word-level bounding boxes to
separate the Akkadian left column from the English right column, then aligns the
noisy OCR output to the gold transliteration lines.

**Input:**  `data/gold/veenhof/*.pdf` + `data/gold/veenhof/*.txt`
**Output:** `results/ocr_pairs.jsonl`

### How to run on Colab
1. Upload the whole `byt5-akkadian-ocr/` folder to Google Drive
2. Open this notebook in Colab
3. Set `DRIVE_REPO_PATH` in the cell below to match your Drive path
4. Run all cells top to bottom — output writes back to Drive automatically

In [1]:
# ── Colab setup ──────────────────────────────────────────────────────────────
import os, sys

ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # ← Change this to wherever you uploaded the repo folder in Drive
    DRIVE_REPO_PATH = '/content/drive/MyDrive/byt5-akkadian-ocr'

    sys.path.insert(0, DRIVE_REPO_PATH)
    os.chdir(DRIVE_REPO_PATH)

    !apt-get install -y -q tesseract-ocr
    !pip install -q pymupdf pytesseract rapidfuzz Pillow pandas opencv-python-headless
else:
    # Running locally
    sys.path.insert(0, '..')
    os.chdir('..')

print(f'Working directory: {os.getcwd()}')

Working directory: /Users/evanlee/Desktop/Byt5-Akkadian-OCR-Correction


In [2]:
import json
import re
from pathlib import Path

import fitz          # pymupdf
import numpy as np
import pandas as pd
import pytesseract
from PIL import Image

from src.parsing.veenhof_parser import parse_all_chapters
from src.alignment.align import align_ocr_to_gold

## Configuration — adjust if needed

In [3]:
VEENHOF_DIR = Path('data/gold/veenhof')
OUTPUT_PATH = Path('results/ocr_pairs.jsonl')
OUTPUT_PATH.parent.mkdir(exist_ok=True)

RENDER_SCALE     = 2     # 2x zoom for better OCR quality
CONF_THRESHOLD   = 30    # discard words Tesseract rates below this confidence
LEFT_COL_CUTOFF  = 0.52  # words left of this fraction of page width = Akkadian column
MIN_ALIGN_SCORE  = 60    # minimum fuzzy-match score to keep a noisy-gold pair

## Load gold lines from the .txt files

In [4]:
gold_df = parse_all_chapters(VEENHOF_DIR)
print(f'Gold lines loaded: {len(gold_df)} across {gold_df.tablet_id.nunique()} tablets')
gold_df.head(3)

Gold lines loaded: 4026 across 256 tablets


,source,tablet_id,line_id,akkadian_gold
0,veenhof,veenhof_ch10,veenhof_ch10_00000,﻿VII. TRAVELING AND TRAVEL-EXPENSES
1,veenhof,veenhof_ch10,veenhof_ch10_00001,VII. TRAVELING AND TRAVEL-EXPENSES (145-153)
2,veenhof,veenhof_ch10_145. Kt 91/k 424 (1-277-91),veenhof_ch10_00002,[x]* °ma¿-na ší-kam ig-ri ša bi4-il5-/tim


## OCR function — extracts left-column (Akkadian) text from one PDF

Uses `image_to_data` to get word-level bounding boxes. Words whose x-center falls
left of `LEFT_COL_CUTOFF × page_width` are treated as Akkadian; everything else
(English translation column) is discarded.

In [5]:
def pdf_to_akkadian_lines(pdf_path, scale=RENDER_SCALE, conf_threshold=CONF_THRESHOLD, left_col_cutoff=LEFT_COL_CUTOFF):
    doc = fitz.open(pdf_path)
    all_lines = []

    for page in doc:
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale))
        img = Image.fromarray(
            np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        )
        page_width = pix.width

        data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)

        rows = []
        for i, text in enumerate(data['text']):
            text = text.strip()
            if not text or int(data['conf'][i]) < conf_threshold:
                continue
            x_center = data['left'][i] + data['width'][i] / 2
            if x_center / page_width < left_col_cutoff:
                rows.append({
                    'block': data['block_num'][i],
                    'line':  data['line_num'][i],
                    'word':  data['word_num'][i],
                    'text':  text,
                })

        if not rows:
            continue

        df = pd.DataFrame(rows)
        for (_, _), grp in df.sort_values(['block','line','word']).groupby(['block','line']):
            line = ' '.join(grp['text'].tolist()).strip()
            if line:
                all_lines.append(line)

    return all_lines

## Debug helper — visualise the column split on one page

Run this on any chapter before the main loop to verify the column boundary looks right.
Green boxes = Akkadian (kept). Red boxes = English (discarded). Magenta line = split.

If the split is wrong, adjust `LEFT_COL_CUTOFF` above.

In [6]:
import cv2
from IPython.display import display

def visualise_columns(pdf_path, page_num=1):
    doc = fitz.open(pdf_path)
    pix = doc[page_num].get_pixmap(matrix=fitz.Matrix(RENDER_SCALE, RENDER_SCALE))
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n).copy()
    page_width = pix.width

    data = pytesseract.image_to_data(Image.fromarray(img), output_type=pytesseract.Output.DICT)
    for i, text in enumerate(data['text']):
        if not text.strip() or int(data['conf'][i]) < 0:
            continue
        x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
        x_center = x + w / 2
        color = (0, 180, 0) if x_center / page_width < LEFT_COL_CUTOFF else (0, 0, 200)
        cv2.rectangle(img, (x, y), (x+w, y+h), color, 2)

    split_x = int(page_width * LEFT_COL_CUTOFF)
    cv2.line(img, (split_x, 0), (split_x, pix.height), (255, 0, 255), 3)
    display(Image.fromarray(img))

# Uncomment to debug — pick any PDF and page:
# visualise_columns(VEENHOF_DIR / '4. I. Six Basic Documents, 1-6.pdf', page_num=1)

## Main loop — OCR all 17 chapters and align to gold

In [7]:
all_pairs = []
yield_log = []

for pdf_path in sorted(VEENHOF_DIR.glob('*.pdf')):
    ch_match = re.match(r'(\d+)', pdf_path.stem)
    if not ch_match:
        print(f'  Skipping {pdf_path.name} — no chapter number in filename')
        continue

    ch_tag = f'veenhof_ch{int(ch_match.group(1)):02d}'
    chapter_gold = gold_df[gold_df['tablet_id'].str.startswith(ch_tag)]['akkadian_gold'].tolist()

    if not chapter_gold:
        print(f'  No gold lines for {ch_tag}, skipping')
        continue

    print(f'Processing {pdf_path.name}  ({len(chapter_gold)} gold lines)...')
    ocr_lines = pdf_to_akkadian_lines(pdf_path)
    aligned   = align_ocr_to_gold(ocr_lines, chapter_gold, min_score=MIN_ALIGN_SCORE)
    yield_pct = 100 * len(aligned) / len(chapter_gold)
    print(f'  → {len(aligned)}/{len(chapter_gold)} aligned  ({yield_pct:.0f}%)')

    for noisy, gold, score in aligned:
        all_pairs.append({'noisy': noisy, 'gold': gold, 'score': round(score, 1), 'source': ch_tag})

    yield_log.append({'chapter': ch_tag, 'gold': len(chapter_gold),
                      'aligned': len(aligned), 'yield_pct': round(yield_pct, 1)})

print(f'\nTotal OCR pairs: {len(all_pairs)}')

  No gold lines for veenhof_ch01, skipping
Processing 10. VII.docx.pdf  (164 gold lines)...
  → 99/164 aligned  (60%)
Processing 11. VIII. Lamassatum, 154-164.pdf  (185 gold lines)...
  → 131/185 aligned  (71%)
Processing 12. IX. The children of Elamma, 165-178.pdf  (232 gold lines)...
  → 157/232 aligned  (68%)
Processing 13. X. Ištar-lamassi, 179-188.pdf  (193 gold lines)...
  → 134/193 aligned  (69%)
Processing 14. XI. Ir'am-Aššur, 189-205.pdf  (312 gold lines)...
  → 200/312 aligned  (64%)
Processing 15. XII. Šalimma, 206-210.pdf  (88 gold lines)...
  → 37/88 aligned  (42%)
Processing 16. XIII.docx.pdf  (400 gold lines)...
  → 256/400 aligned  (64%)
Processing 17. XIV. Šu-Ištar, 235-245.pdf  (152 gold lines)...
  → 93/152 aligned  (61%)
  No gold lines for veenhof_ch02, skipping
  No gold lines for veenhof_ch03, skipping
Processing 4. I. Six Basic Documents, 1-6.pdf  (107 gold lines)...
  → 88/107 aligned  (82%)
Processing 5. II. Caravan Documents, 7-53.pdf  (917 gold lines).

## Inspect yield by chapter

Chapters below ~40% yield probably have a layout that doesn't match `LEFT_COL_CUTOFF`.
Use `visualise_columns()` above to debug those chapters specifically.

In [8]:
yield_df = pd.DataFrame(yield_log)
print(yield_df.to_string(index=False))

if all_pairs:
    scores = [p['score'] for p in all_pairs]
    print(f'\nAlignment score — mean: {sum(scores)/len(scores):.1f}, min: {min(scores)}, max: {max(scores)}')
    print('\nSample pairs (5 random):')
    import random
    for p in random.sample(all_pairs, min(5, len(all_pairs))):
        print(f'  gold : {p["gold"]}')
        print(f'  noisy: {p["noisy"]}')
        print()

     chapter  gold  aligned  yield_pct
veenhof_ch10   164       99       60.4
veenhof_ch11   185      131       70.8
veenhof_ch12   232      157       67.7
veenhof_ch13   193      134       69.4
veenhof_ch14   312      200       64.1
veenhof_ch15    88       37       42.0
veenhof_ch16   400      256       64.0
veenhof_ch17   152       93       61.2
veenhof_ch04   107       88       82.2
veenhof_ch05   917      604       65.9
veenhof_ch06   400      320       80.0
veenhof_ch07   323      181       56.0
veenhof_ch08   328      228       69.5
veenhof_ch09   225      143       63.6

Alignment score — mean: 79.7, min: 60.0, max: 100.0

Sample pairs (5 random):
  gold : DUMU  A-mur-Ištar
  noisy: DUMU A-zu-a 2

  gold : 1½ GÍN.TA : i-na ITU.1.KAM  ú-ṣa-áb
  noisy: 1% GIN.TA : i-na ITU. 1.KAM

  gold : a-na  4 ha-am-ša-tim
  noisy: 15 ha-am-Sa-tim will

  gold : Im-dí-DINGIR i-šu
  noisy: Im-di-DINGIR i-Su

  gold : DUMU  Qá-a-a-tim
  noisy: DUMU Q tim son



## Save to results/ocr_pairs.jsonl

In [9]:
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for pair in all_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + '\n')

print(f'Saved {len(all_pairs)} pairs to {OUTPUT_PATH}')
print('This file is already in results/ — the training notebook will pick it up automatically.')

Saved 2671 pairs to results/ocr_pairs.jsonl
This file is already in results/ — the training notebook will pick it up automatically.
